# 05 - Synthetic Control: Robustness Checks
**Owner:** Dhananjay · **Week 3**

This notebook checks whether the estimated post-intervention revenue gap is sensitive to timing, data frequency, and donor choice. It uses the project's **synthetic/simulated dataset**, so findings are methodological demonstrations rather than evidence about a real business.

**Primary questions**
1. Does the main weekly estimate reproduce?
2. What happens when we assign false intervention dates in the pre-treatment period?
3. Is the estimate similar when calculated from daily rather than weekly data?
4. How sensitive is the result to using each donor region alone?
5. Does the treated series visually track its synthetic counterfactual before treatment?


## 1. Setup and data checks

The intervention date is 1 May 2025. The synthetic control weights are fitted using pre-treatment observations only, are constrained to be non-negative, and sum to one. Weekly rows are retained only when the week contains seven daily observations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

DATA_PATH = "project1_algorithmic_pricing_synthetic_dataset.csv"
TREATMENT_DATE = pd.Timestamp("2025-05-01")
TREATED = "North"
DONORS = ["South", "West"]

df = pd.read_csv(DATA_PATH, parse_dates=["date"])
daily = df.pivot(index="date", columns="region", values="revenue").sort_index()
weekly_all = daily.resample("W-SUN").sum()
weekly = weekly_all[daily.resample("W-SUN").size() == 7]

print(f"Dataset rows: {len(df):,}")
print(f"Daily dates: {daily.index.min().date()} to {daily.index.max().date()}")
print(f"Regions: {', '.join(daily.columns)}")
print(f"Complete weekly observations: {len(weekly)}")
print(f"Intervention date: {TREATMENT_DATE.date()}")

assert TREATED in daily.columns, f"Missing treated region: {TREATED}"
assert all(region in daily.columns for region in DONORS), "One or more donor regions are missing"
assert daily.index.is_monotonic_increasing
assert daily[[TREATED] + DONORS].notna().all().all(), "Revenue contains missing values"


## 2. Estimation functions

`fit_weights` uses only rows strictly before the cutoff. `estimate_effect` compares mean treated revenue with the weighted synthetic series during the selected post-cutoff window. The normalized pre-period RMSE is reported as a fit diagnostic, not as a causal-validity test.

**Timing handling:** the weekly series is labelled by week-ending Sunday. Since the intervention begins on Thursday 1 May, the week ending 4 May includes both pre- and post-intervention days. For the main weekly estimate and donor-sensitivity checks, this mixed week is excluded from the post-period; the weekly post-period therefore starts with the week ending 11 May. The daily estimate retains 1 May as its intervention date.

In [ ]:
def fit_weights(data, treated, donors, cutoff):
    pre_mask = data.index < cutoff
    pre = data.loc[pre_mask, [treated] + donors].dropna()
    if len(pre) < 3:
        raise ValueError(f"Not enough pre-treatment rows before {cutoff.date()}")

    X = pre[donors].to_numpy(dtype=float)
    y = pre[treated].to_numpy(dtype=float)
    scale = max(float(np.mean(np.abs(y))), 1e-12)

    result = minimize(
        lambda w: np.mean(((y - X @ w) / scale) ** 2),
        x0=np.full(len(donors), 1 / len(donors)),
        bounds=[(0, 1)] * len(donors),
        constraints={"type": "eq", "fun": lambda w: np.sum(w) - 1},
        method="SLSQP",
    )
    if not result.success:
        raise RuntimeError(f"Weight optimization failed: {result.message}")
    return pd.Series(result.x, index=donors, name="weight")


def estimate_effect(data, treated, donors, cutoff, end=None, post_start=None):
    weights = fit_weights(data, treated, donors, cutoff)
    synthetic = data[donors].mul(weights, axis=1).sum(axis=1)

    pre_mask = data.index < cutoff
    effective_post_start = cutoff if post_start is None else post_start
    post_mask = data.index >= effective_post_start
    if end is not None:
        post_mask &= data.index < end

    if not pre_mask.any() or not post_mask.any():
        raise ValueError(f"Empty pre- or post-period for cutoff {cutoff.date()}")

    treated_pre = data.loc[pre_mask, treated]
    synthetic_pre = synthetic.loc[pre_mask]
    denominator = float(treated_pre.mean())
    pre_rmse_pct = (
        np.sqrt(np.mean((treated_pre - synthetic_pre) ** 2)) / max(abs(denominator), 1e-12) * 100
    )

    treated_post = data.loc[post_mask, treated]
    synthetic_post = synthetic.loc[post_mask]
    effect_pct = (
        (treated_post.mean() - synthetic_post.mean())
        / max(abs(float(synthetic_post.mean())), 1e-12)
        * 100
    )

    return {
        "effect_pct": float(effect_pct),
        "pre_rmse_pct": float(pre_rmse_pct),
        "weights": weights,
        "synthetic": synthetic,
        "pre_mask": pre_mask,
        "post_mask": post_mask,
    }


WEEKLY_BOUNDARY_WEEK = pd.Timestamp("2025-05-04")
WEEKLY_POST_START = pd.Timestamp("2025-05-05")  # excludes the week ending 4 May
main_weekly = estimate_effect(weekly, TREATED, DONORS, TREATMENT_DATE, post_start=WEEKLY_POST_START)
main_daily = estimate_effect(daily, TREATED, DONORS, TREATMENT_DATE)

print(f"Weekly mixed boundary week excluded: {WEEKLY_BOUNDARY_WEEK.date()}")
print(f"Main result (weekly): {main_weekly['effect_pct']:.1f}% mean revenue gap")
print(f"Weekly pre-period normalized RMSE: {main_weekly['pre_rmse_pct']:.1f}%")
print(f"Daily result: {main_daily['effect_pct']:.1f}% mean revenue gap")
print(f"Daily pre-period normalized RMSE: {main_daily['pre_rmse_pct']:.1f}%")
print("\nWeekly synthetic-control weights:")
display(main_weekly["weights"].to_frame())


## 3. Test A — Placebo intervention dates

For each false intervention date before 1 May, fit weights using observations before that date and estimate the gap after it but still before the real intervention. A large placebo gap may indicate that the method can produce apparent effects even when no intervention has occurred.

These checks are descriptive: with only two donor regions and a short pre-period, they do not establish statistical significance.

In [ ]:
placebo_dates = [
    pd.Timestamp("2025-02-01"),
    pd.Timestamp("2025-03-01"),
    pd.Timestamp("2025-04-01"),
]

placebo_rows = []
for placebo_date in placebo_dates:
    try:
        result = estimate_effect(
            weekly[weekly.index < TREATMENT_DATE],
            TREATED,
            DONORS,
            placebo_date,
            end=TREATMENT_DATE,
        )
        placebo_rows.append({
            "placebo_date": placebo_date.date().isoformat(),
            "estimated_gap_pct": result["effect_pct"],
            "pre_rmse_pct": result["pre_rmse_pct"],
        })
    except ValueError as exc:
        placebo_rows.append({
            "placebo_date": placebo_date.date().isoformat(),
            "estimated_gap_pct": np.nan,
            "pre_rmse_pct": np.nan,
            "note": str(exc),
        })

placebo_results = pd.DataFrame(placebo_rows)
display(placebo_results.style.format({
    "estimated_gap_pct": "{:.2f}%",
    "pre_rmse_pct": "{:.2f}%"
}, na_rep="N/A"))


## 4. Test B — Donor sensitivity

There are only two potential donor regions, South and West. Re-estimate using each region alone, then compare with the two-donor weighted result. Single-donor models are sensitivity checks, not necessarily better counterfactuals.

In [ ]:
donor_rows = []
for donor_set, label in [
    (DONORS, "South + West"),
    (["South"], "South only"),
    (["West"], "West only"),
]:
    result = estimate_effect(weekly, TREATED, donor_set, TREATMENT_DATE, post_start=WEEKLY_POST_START)
    donor_rows.append({
        "donor_specification": label,
        "estimated_gap_pct": result["effect_pct"],
        "pre_rmse_pct": result["pre_rmse_pct"],
        "weights": ", ".join(f"{k}={v:.3f}" for k, v in result["weights"].items()),
    })

donor_sensitivity = pd.DataFrame(donor_rows)
display(donor_sensitivity.style.format({
    "estimated_gap_pct": "{:.2f}%",
    "pre_rmse_pct": "{:.2f}%"
}))


## 5. Test C — Actual versus synthetic trajectory

The plot helps assess whether the synthetic series tracks North before treatment and how the gap evolves afterwards. Visual fit is useful but cannot, by itself, prove the identifying assumptions.

In [ ]:
weekly_synthetic = main_weekly["synthetic"]
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(weekly.index, weekly[TREATED], label="North (treated)", linewidth=2)
ax.plot(weekly.index, weekly_synthetic, label="Synthetic North", linewidth=2, linestyle="--")
ax.axvline(TREATMENT_DATE, color="black", linestyle=":", label="Intervention (1 May 2025)")
ax.set_title("North revenue vs synthetic counterfactual (weekly)")
ax.set_xlabel("Week ending")
ax.set_ylabel("Revenue")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

gap = weekly[TREATED] - weekly_synthetic
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(weekly.index, gap, label="North minus synthetic")
ax.axhline(0, color="black", linewidth=1)
ax.axvline(TREATMENT_DATE, color="black", linestyle=":", label="Intervention")
ax.set_title("Weekly revenue gap")
ax.set_xlabel("Week ending")
ax.set_ylabel("Revenue gap")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## 6. Summary and interpretation

The cells below summarize the estimated gaps and diagnostics. Interpret these as estimated differences in this synthetic dataset, not as proven causal effects.

**Limitations to report**
- The dataset is simulated and does not represent actual company transactions.
- Only two donor regions are available; donor-pool-based inference is weak.
- A good pre-period fit does not guarantee that the treated region would have followed the synthetic control after the intervention without treatment.
- Placebo checks are limited in number and are diagnostic, not formal significance tests.
- The weekly bucket ending 4 May straddles the 1 May intervention; compare with the daily estimate when interpreting timing.
- Revenue alone does not identify the mechanism. Price, demand, promotions, marketing, and competitor actions may also matter.


In [ ]:
summary = pd.DataFrame([
    {
        "analysis": "Main weekly estimate",
        "estimated_revenue_gap_pct": main_weekly["effect_pct"],
        "pre_period_normalized_rmse_pct": main_weekly["pre_rmse_pct"],
    },
    {
        "analysis": "Main daily estimate",
        "estimated_revenue_gap_pct": main_daily["effect_pct"],
        "pre_period_normalized_rmse_pct": main_daily["pre_rmse_pct"],
    },
])
display(summary.style.format({
    "estimated_revenue_gap_pct": "{:.2f}%",
    "pre_period_normalized_rmse_pct": "{:.2f}%"
}))

print("Reminder: these are descriptive estimates on simulated data, not proof of a real-world causal effect.")
